# Stacks and queues: bracket checks and level-order processing

**STUDY OBJECTIVES**

By the end of this lesson, we should be able to:

- Know the difference between a stack and a queue.
- Choose which one fits a problem and explain why.
- Use a stack to check whether brackets are balanced.
- Use a queue to process items in arrival order and visit a tree level by level.

## 1. Why these data structures matter

Many problems involve items waiting to be processed. The important question is often:

> **Which item should be handled next?**

A **stack** handles the most recently added item first. A **queue** handles the earliest added item first.

Imagine two situations:

- You put plates on top of a pile. The plate you can take first is the one you put down most recently.
- People join a line. The person who joined first is served first.

These are the ideas behind **LIFO** and **FIFO**:

- **LIFO** means *Last In, First Out*: a stack.
- **FIFO** means *First In, First Out*: a queue.

Choosing the right structure can make a solution easier to understand and more efficient.

## 2. Stacks

A stack supports two main operations:

- **Push:** add an item to the top.
- **Pop:** remove the item at the top.

You can also **peek** at the top item without removing it.

### A stack using a Python list

Python lists work well as stacks when you add and remove from the end.

In [ ]:
books_to_put_away = []

books_to_put_away.append("Database Systems")  # push
books_to_put_away.append("Python Basics")     # push
books_to_put_away.append("Web APIs")          # push

print(books_to_put_away[-1])  # peek: Web APIs

next_book = books_to_put_away.pop()  # pop
print(next_book)                     # Web APIs
print(books_to_put_away)             # two books remain

The last book added is the first one removed.

For a list used as a stack:

- `append(item)` adds to the top.
- `pop()` removes from the top.
- `stack[-1]` looks at the top without removing it.

These end-of-list operations usually take **O(1)** time: their cost stays roughly constant as the stack grows.

### When a stack is useful

A stack is a good fit when the newest unfinished item should be handled first. Common examples include:

- Matching brackets in code or configuration files.
- Undo and redo features.
- Tracking nested function calls.
- Depth-first search through a graph or tree.
- Reversing a sequence.

### What happens when you pop an empty stack?

Calling `pop()` on an empty list raises an `IndexError`. A good solution checks before popping.

In [2]:
stack = []

if stack:
    item = stack.pop()
else:
    print("There is nothing to remove.")

There is nothing to remove.


For the bracket-checking problem, an empty stack at the wrong time tells us that the input is invalid.

## 3. Bracket matching with a stack

Suppose we want to check whether a string contains correctly matched brackets.

Examples:

```text
{[()]}     balanced
()[]{}     balanced
([)]       not balanced
((())      not balanced
())        not balanced
```

A closing bracket must match the **most recent unmatched opening bracket**. That is exactly what a stack provides.

### The algorithm

Read the characters from left to right:

1. When you see an opening bracket—`(`, `[`, or `{`—push it onto the stack.
2. When you see a closing bracket:
   - If the stack is empty, there is no opening bracket to match it, so return `False`, meaning the brackets are not balanced or there's a mismatch.
   - Otherwise, pop the most recent opening bracket.
   - If the popped bracket is the wrong type, return `False`, (not balanced or there's a mismatch).
3. After reading the whole string, return `True` only if the stack is empty.

Why check that the stack is empty at the end? Each opening bracket needs a closing bracket. Any items left on the stack were never closed.

### Runnable implementation

In [1]:
def brackets_are_balanced(text: str) -> bool:
    """Return True if (), [] and {} are correctly matched in text."""
    opening_brackets = "([{"
    matching_opening = {
        ")": "(",
        "]": "[",
        "}": "{",
    }
    stack: list[str] = []
    for i in text:
        if i in opening_brackets:
            stack.append(i)
        elif i in matching_opening:
            if not stack:
                return False
            most_recent_opening = stack.pop()
            if most_recent_opening != matching_opening[i]:
                return False
    return not stack
examples = [
    "{[()]})",
]

for example in examples:
    print(example, brackets_are_balanced(example))

{[()]}) False


Try it:

In [4]:
examples = [
    "{[()]}",
    "()[]{}",
    "([)]",
    "((())",
    "())",
]

for example in examples:
    print(example, brackets_are_balanced(example))

{[()]} True
()[]{} True
([)] False
((()) False
()) False


Expected output:

```text
{[()]} True
()[]{} True
([)] False
((()) False
()) False
```

### Trace an example by hand

For the input `"{[()]}"`:

| Character | Action | Stack after action |
|---|---|---|
| `{` | Push `{` | `{` |
| `[` | Push `[` | `{ [` |
| `(` | Push `(` | `{ [ (` |
| `)` | Pop `(`; it matches | `{ [` |
| `]` | Pop `[`; it matches | `{` |
| `}` | Pop `{`; it matches | empty |

The input is balanced because no mismatch occurred and the stack is empty at the end.

Now consider `"([)]"`:

| Character | Action | Stack after action |
|---|---|---|
| `(` | Push `(` | `(` |
| `[` | Push `[` | `( [` |
| `)` | Pop `[`; expected `(` | mismatch |

The string contains the right number of opening and closing brackets, but the nesting order is wrong. Counting each bracket type separately would miss that. The stack remembers the order needed to detect it.

### Complexity

For a string of `n` characters:

- **Time:** O(n), because each character is inspected once.
- **Extra space:** O(n) in the worst case, because all characters could be opening brackets.

The stack uses extra memory, but it gives us the information needed to check nesting correctly.

### A focused failure experiment

A useful test is to deliberately remove one of the checks and see what breaks. For example, remove this check:

```python
if not stack:
    return False
```

Then try `brackets_are_balanced(")")`.

With the check present, it returns `False`. Without the check, the code tries to pop an empty stack and raises `IndexError`. Run the cell below to see both outcomes.

Being able to explain this failure shows why the check is part of the algorithm.

In [ ]:
def brackets_without_the_empty_check(text: str) -> bool:
    """The same algorithm with the empty-stack check removed, for comparison."""
    opening_brackets = "([{"
    matching_opening = {")": "(", "]": "[", "}": "{"}

    stack: list[str] = []

    for character in text:
        if character in opening_brackets:
            stack.append(character)
        elif character in matching_opening:
            most_recent_opening = stack.pop()  # no empty-stack check here
            if most_recent_opening != matching_opening[character]:
                return False

    return not stack


print("with the check:   ", brackets_are_balanced(")"))

try:
    brackets_without_the_empty_check(")")
except IndexError as error:
    print("without the check:", type(error).__name__, "-", error)

## 4. Queues

A queue adds items at one end and removes them from the other:

- **Enqueue:** add an item to the back.
- **Dequeue:** remove the item at the front.
- **Peek:** inspect the item at the front.

The first item added is the first item removed.

### Why not use `list.pop(0)`?

A list can represent a queue, but removing its first item is inefficient.

In [ ]:
# Imagine Class 2 being a list
# If for instance, the index number represents how ready each member of the class are: index 0 being at the top,

class_2_members = ["Emeto", "Fawo", "Young-beck"]

# If Joey come to pick them up one by one for internship:

first_intern = class_2_members.pop(0)
print(first_intern)      # Emeto
print(class_2_members)   # Fawo and Young-beck remain

When the first item is removed, Python has to shift the other items along. For a long queue, repeating this work can be costly.

Instead, using `collections.deque` is better. It is designed for adding and removing items at either end.

### A queue using `deque`

In [2]:
from collections import deque

class_2_members = deque()

class_2_members.append("Emeto")    # enqueue
class_2_members.append("Fawo")
class_2_members.append("Youn-beck")

next_intern = class_2_members.popleft()  # dequeue
next_person = class_2_members[0]
print(next_person)
print(next_intern)                       # Emeto
print(class_2_members)                   # Fawo, Young-beck

Fawo
Emeto
deque(['Fawo', 'Youn-beck'])


For `deque`:

- `append(item)` adds to the back.
- `popleft()` removes from the front.
- `queue[0]` looks at the front without removing it.

Adding at the back and removing from the front are both typically **O(1)** operations.

### Process a queue until it is empty

In [3]:
tasks = deque(["index books", "check loan", "send reminder"])

while tasks: # While tasks is not empty
    task = tasks.popleft()
    print("Processing:", task)

Processing: index books
Processing: check loan
Processing: send reminder


The tasks are processed in the same order they were added.

### When a queue is useful

A queue is a good fit when the earliest waiting item should be handled first. Examples include:

- A returns desk serving members in arrival order.
- Jobs waiting for a worker.
- Messages waiting to be processed.
- Breadth-first search and level-order tree traversal.
- A simulation of customers or requests arriving over time.

A queue preserves arrival order. The trade-off is that the program may need to hold many waiting items in memory.

## 5. Level-order processing

A **tree** is a structure made of nodes connected in parent–child relationships. Level-order traversal visits all nodes at one depth before moving to the next.

For example:

```text
        A
       / \
      B   C
     / \   \
    D   E   F
```

Level order is:

```text
A, B, C, D, E, F
```

The queue makes this order natural:

1. Put the root node in the queue.
2. Remove the node at the front and process it.
3. Add its children to the back.
4. Repeat until the queue is empty.

### A small tree implementation

In [4]:
from collections import deque
from dataclasses import dataclass


@dataclass
class Node:
    value: str
    left: "Node | None" = None
    right: "Node | None" = None


def level_order(root: Node | None) -> list[str]:
    """Return the node values from top to bottom, left to right."""
    if root is None:
        return []

    values: list[str] = []
    queue: deque[Node] = deque([root])

    while queue: # While queue is not empty,
        current = queue.popleft()
        values.append(current.value)

        if current.left is not None:
            queue.append(current.left)

        if current.right is not None:
            queue.append(current.right)

    return values


tree = Node(
    "A",
    left=Node("B", left=Node("D"), right=Node("E")),
    right=Node("C", right=Node("F")),
)

print(level_order(tree))

['A', 'B', 'C', 'D', 'E', 'F']


Expected output:

```text
['A', 'B', 'C', 'D', 'E', 'F']
```

### Why does this produce?

After processing `A`, the queue contains `B, C`. After processing `B`, we add its children after `C`, so the queue contains `C, D, E`. The older waiting node `C` is handled before the children added by `B`.

That FIFO behavior keeps the traversal moving across the current level before going deeper.

### Complexity

For a tree with `n` nodes:

- **Time:** O(n), because each node is added and removed once.
- **Extra space:** O(w), where `w` is the maximum number of nodes waiting at one time. In the worst case, this can be O(n).

## 6. Stack or queue?

Use this quick decision:

- Choose a **stack** when the newest item should be handled first, or when you need to remember nested, unfinished work.
- Choose a **queue** when the oldest waiting item should be handled first, or when you need to process items level by level.

A bracket checker needs a stack because the most recent opening bracket must be matched first. A returns desk needs a queue because members should generally be served in the order they arrive.

Don't choose a structure just because it is familiar. Start with the order the problem requires.

## 7. Final recap

- **Stack = LIFO:** last in, first out. Use it for bracket matching and nested work.
- **Queue = FIFO:** first in, first out. Use it for waiting lines and level-order processing.
- A bracket checker pushes opening brackets and checks each closing bracket against the most recent opening one.
- A level-order traversal queues children so older waiting nodes are processed first.
- For both examples, each item is processed a limited number of times, so the algorithms run in O(n) time.

**Self-check:** Can you explain why `"([)]"` fails even though it has two opening and two closing brackets? If you can answer that and walk through the queue for the tree example, you've understood the core of your Week 2 topic.